In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load
df = pd.read_csv(r"D:\SHAP-Guided Bi-LSTM Autoencoder Framework for Slow-Rate DDoS and Zero-Day Attack Detection\Datasets\Wednesday-workingHours.pcap_ISCX.csv")  # adjust path
df.columns = df.columns.str.strip()
df = df.drop(columns=['Fwd Header Length.1'])

# Clean NaN/Inf
nan_mask = df.isnull().any(axis=1)
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_mask = np.isinf(df[numeric_cols]).any(axis=1)
df_clean = df[~(nan_mask | inf_mask)].reset_index(drop=True)

# Filter to target labels
target_labels = ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest']
df_filtered = df_clean[df_clean['Label'].isin(target_labels)].reset_index(drop=True)

# Split
X = df_filtered.drop(columns=['Label'])
y = df_filtered['Label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_test shape:", X_test.shape)

X_test shape: (90196, 77)


In [4]:
import joblib

scaler = joblib.load(r"C:\Users\SATHISH\sathish project\scaler_slowrate.pkl")  # the one from Colab, already fitted on BENIGN

# Drop the same zero-variance columns we dropped before
zero_var_col_names = ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count',
                       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
                       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

X_test = X_test.drop(columns=zero_var_col_names, errors='ignore')

# Use transform only - this scaler is already fitted, do NOT re-fit
X_test_scaled = scaler.transform(X_test)

X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
print("X_test_reshaped shape:", X_test_reshaped.shape)

C:\Users\SATHISH\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


X_test_reshaped shape: (90196, 67, 1)


In [5]:
from tensorflow.keras.models import load_model

autoencoder = load_model(r"C:\Users\SATHISH\sathish project\bilstm_autoencoder_slowrate.keras")
autoencoder.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 67, 1)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_4 (Bidirectional)      │ (None, 67, 128)             │          33,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_5 (Bidirectional)      │ (None, 64)                  │          41,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector_1 (RepeatVector)       │ (None, 67, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_6 (Bidirectional)      │ (None, 67, 64)              │          24,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_7 (Bidirectional)      │ (None, 67, 128)             │          66,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_1 (TimeDistributed) │ (None, 67, 1)               │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 498,053 (1.90 MB)

 Trainable params: 166,017 (648.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 332,036 (1.27 MB)

In [6]:
# Get reconstructions for the full test set (BENIGN + Slowloris + Slowhttptest mixed)
X_test_pred = autoencoder.predict(X_test_reshaped)

# Reconstruction error per sample = Mean Squared Error between original and reconstructed
reconstruction_error = np.mean(np.square(X_test_reshaped - X_test_pred), axis=(1, 2))

# Attach errors back to their true labels for comparison
error_df = pd.DataFrame({
    'reconstruction_error': reconstruction_error,
    'true_label': y_test.values
})

print(error_df.groupby('true_label')['reconstruction_error'].describe())

2819/2819 ━━━━━━━━━━━━━━━━━━━━ 311s 109ms/step
                    count      mean       std       min       25%       50%  \
true_label                                                                    
BENIGN            87937.0  0.139198  7.772696  0.003926  0.006178  0.014492   
DoS Slowhttptest   1100.0  1.543931  4.667031  0.012341  0.757050  0.895055   
DoS slowloris      1159.0  1.258618  1.560300  0.012547  0.200488  0.656812   

                       75%          max  
true_label                               
BENIGN            0.055871  2262.778415  
DoS Slowhttptest  0.895849    95.086423  
DoS slowloris     2.017957    11.724383  


In [7]:
benign_errors = error_df[error_df['true_label'] == 'BENIGN']['reconstruction_error']
print("Number of BENIGN rows with error > 10:", (benign_errors > 10).sum())
print("Number of BENIGN rows with error > 100:", (benign_errors > 100).sum())

Number of BENIGN rows with error > 10: 122
Number of BENIGN rows with error > 100: 3
